In [ ]:
import json
import re
import re
import pandas as pd
from pathlib import Path

def json_to_df(path):

    with open(path, 'r') as file:
        data = json.load(file)

    titles = []
    contexts = []
    answers_list = []

    L = data['data']
    for i in range(len(L)):
        contract = L[i]
        title = contract['title']
        paragraphs = contract['paragraphs'][0]
        context = paragraphs['context'].strip()
        answers = {}
        for item in paragraphs['qas']:
            answer = {}
            id = item['id'].strip()
            answer['id'] = id
            answer['is_impossible'] = item['is_impossible']
            question = item['question'].strip()
            questionStart = question.find('"')
            questionEnd = question[questionStart+1:].find('"') + questionStart+1
            question = question[questionStart+1:questionEnd]
            answer['question'] = question
            if item['answers'] != []:
                answer_text = item['answers'][0]['text'].strip()
                answer_start = item['answers'][0]['answer_start']
                answer['text'] = answer_text
            answers[answer_start] = answer
        titles.append(title)
        contexts.append(context)
        answers_list.append(answers)

    d = {'title': titles,
         'context': contexts,
         'answers': answers_list,
        }

    return pd.DataFrame(d)

project_root = Path.cwd().parent
df_train = json_to_df(project_root / 'data/train_separate_questions.json')
print(df_train.shape)
df_train.head()

(408, 3)


,title,context,answers
0,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,EXHIBIT 10.6\n\n ...,{44: {'id': 'LIMEENERGYCO_09_09_1999-EX-10-DIS...
1,"WHITESMOKE,INC_11_08_2011-EX-10.26-PROMOTION A...",Exhibit 10.26 CONFIDENTIAL TREATMENT HAS BE...,"{307: {'id': 'WHITESMOKE,INC_11_08_2011-EX-10...."
2,NELNETINC_04_08_2020-EX-1-JOINT FILING AGREEMENT,Exhibit 1\n\nJOINT FILING AGREEMENT\n\nThe und...,{11: {'id': 'NELNETINC_04_08_2020-EX-1-JOINT F...
3,ADAMSGOLFINC_03_21_2005-EX-10.17-ENDORSEMENT A...,REDACTED COPY\n\nCONFIDENTIAL TREATMENT REQUES...,{171: {'id': 'ADAMSGOLFINC_03_21_2005-EX-10.17...
4,"KIROMICBIOPHARMA,INC_05_11_2020-EX-10.23-CONSU...",Exhibit 10.23 Corporate Address Fannin South P...,"{135: {'id': 'KIROMICBIOPHARMA,INC_05_11_2020-..."


In [ ]:
#basic chunking

def chunk_text_to_200_words(text, chunk_size=200):
    """Split contract text into 200-word chunks and keep their exact character ranges."""
    import re

    if text is None or pd.isna(text):
        return []

    text = str(text)
    matches = list(re.finditer(r"\S+", text))
    if not matches:
        return []

    chunks = []
    for i in range(0, len(matches), chunk_size):
        chunk_matches = matches[i:i + chunk_size]
        start = chunk_matches[0].start()
        end = chunk_matches[-1].end()
        chunks.append({
            'text': text[start:end],
            'start': start,
            'end': end,
        })

    return chunks


def chunk_contract_df(df, text_col='context', chunk_size=200):
    """Expand a contract dataframe into per-chunk rows, keeping only labels whose answer_start falls within the chunk."""
    rows = []

    for _, row in df.iterrows():
        answers = row.get('answers', {}) or {}
        chunks = chunk_text_to_200_words(row[text_col], chunk_size=chunk_size)

        for chunk_index, chunk in enumerate(chunks):
            chunk_labels = []
            for answer_start, answer in answers.items():
                answer_start = int(answer_start)
                if chunk['start'] <= answer_start < chunk['end']:
                    question = answer.get('question')
                    if question is not None:
                        chunk_labels.append(question)

            item = row.to_dict()
            item['chunk_text'] = chunk['text']
            item['chunk_index'] = chunk_index
            item['chunk_start'] = chunk['start']
            item['chunk_end'] = chunk['end']
            item['labels'] = chunk_labels
            item['label'] = chunk_labels[0] if len(chunk_labels) == 1 else None
            rows.append(item)

    return pd.DataFrame(rows)


# Example usage:
train_chunks = chunk_contract_df(df_train, text_col='context', chunk_size=200)
train_chunks[['title', 'labels', 'chunk_index', 'chunk_text']].head()

,title,labels,chunk_index,chunk_text
0,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,"[Document Name, Parties, Parties, Parties, Par...",0,EXHIBIT 10.6\n\n ...
1,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,"[Exclusivity, License Grant]",1,and sell and distribute Products solely for...
2,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,"[Expiration Date, Source Code Escrow]",2,exclusive distributor in that market; oth...
3,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,"[Notice Period To Terminate Renewal, Minimum C...",3,which the Company delivers to Distributor t...
4,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,"[Minimum Commitment, Joint Ip Ownership]",4,subdistributors the ...


In [23]:
all_labels = set()
for labels in train_chunks['labels']:
    for label in labels:
        all_labels.add(label)
all_labels

{'Affiliate License-Licensee',
 'Affiliate License-Licensor',
 'Agreement Date',
 'Anti-Assignment',
 'Audit Rights',
 'Cap On Liability',
 'Change Of Control',
 'Competitive Restriction Exception',
 'Covenant Not To Sue',
 'Document Name',
 'Effective Date',
 'Exclusivity',
 'Expiration Date',
 'Governing Law',
 'Insurance',
 'Ip Ownership Assignment',
 'Irrevocable Or Perpetual License',
 'Joint Ip Ownership',
 'License Grant',
 'Liquidated Damages',
 'Minimum Commitment',
 'Most Favored Nation',
 'No-Solicit Of Customers',
 'No-Solicit Of Employees',
 'Non-Compete',
 'Non-Disparagement',
 'Non-Transferable License',
 'Notice Period To Terminate Renewal',
 'Parties',
 'Post-Termination Services',
 'Price Restrictions',
 'Renewal Term',
 'Revenue/Profit Sharing',
 'Rofr/Rofo/Rofn',
 'Source Code Escrow',
 'Termination For Convenience',
 'Third Party Beneficiary',
 'Uncapped Liability',
 'Unlimited/All-You-Can-Eat-License',
 'Volume Restriction',
 'Warranty Duration'}

In [ ]:
# define categories for classifier

relevant_categories = [
    'Exclusivity',
    'Insurance',
    'Ip Ownership Assignment',
    'Irrevocable Or Perpetual License',
    'Unlimited/All-You-Can-Eat-License',
    'Volume Restriction',
    'Warranty Duration',
]

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.multiclass import OneVsRestClassifier
from sklearn.metrics import classification_report, accuracy_score
from sklearn.preprocessing import MultiLabelBinarizer

In [ ]:
# prepare training data for classifier

train_chunks = train_chunks.copy()
train_chunks = train_chunks[
    train_chunks['labels'].apply(lambda labels: any(label in relevant_categories for label in labels))
].copy()
train_chunks['labels'] = train_chunks['labels'].apply(
    lambda labels: [label for label in labels if label in relevant_categories]
)

X_train = train_chunks['chunk_text'].fillna('').astype(str)
mlb = MultiLabelBinarizer(classes=relevant_categories)
y_train = mlb.fit_transform(train_chunks['labels'])

vectorizer = TfidfVectorizer(stop_words='english', ngram_range=(1, 2), min_df=2)
X_train_vec = vectorizer.fit_transform(X_train)

mlb.classes_

array(['Exclusivity', 'Insurance', 'Ip Ownership Assignment',
       'Irrevocable Or Perpetual License',
       'Unlimited/All-You-Can-Eat-License', 'Volume Restriction',
       'Warranty Duration'], dtype=object)

In [ ]:
# load test data and chunk

project_root = Path.cwd().parent
df_test = json_to_df(project_root / 'data/test.json')

df_test_chunks = chunk_contract_df(df_test, text_col='context', chunk_size=200)
df_test_chunks = df_test_chunks[
    df_test_chunks['labels'].apply(lambda labels: any(label in relevant_categories for label in labels))
].copy()
df_test_chunks['labels'] = df_test_chunks['labels'].apply(
    lambda labels: [label for label in labels if label in relevant_categories]
)

X_test = df_test_chunks['chunk_text'].fillna('').astype(str)
y_test = mlb.transform(df_test_chunks['labels'])

In [ ]:
# Logistic Regression baseline
from sklearn.linear_model import LogisticRegression

logreg = LogisticRegression(max_iter=2000, class_weight='balanced', solver='liblinear')
logreg_classifier = OneVsRestClassifier(logreg)
logreg_classifier.fit(X_train_vec, y_train)
logreg_predictions = logreg_classifier.predict(vectorizer.transform(X_test))

print('Logistic Regression multilabel results:')
print(classification_report(y_test, logreg_predictions, target_names=mlb.classes_, zero_division=0))

Logistic Regression multilabel results:
                                   precision    recall  f1-score   support

                      Exclusivity       0.20      0.40      0.27         5
                        Insurance       0.70      0.67      0.68        24
          Ip Ownership Assignment       0.17      0.17      0.17         6
 Irrevocable Or Perpetual License       0.14      1.00      0.25         1
Unlimited/All-You-Can-Eat-License       0.67      0.33      0.44         6
               Volume Restriction       0.65      0.48      0.55        23
                Warranty Duration       0.62      0.56      0.59        32

                        micro avg       0.54      0.53      0.53        97
                        macro avg       0.45      0.52      0.42        97
                     weighted avg       0.59      0.53      0.55        97
                      samples avg       0.45      0.53      0.48        97



In [ ]:
#LinearSVC model

from sklearn.svm import LinearSVC

svc = LinearSVC(class_weight='balanced')
svc_classifier = OneVsRestClassifier(svc)
svc_classifier.fit(X_train_vec, y_train)
svc_predictions = svc_classifier.predict(vectorizer.transform(X_test))

print('LinearSVC multilabel results:')
print(classification_report(y_test, svc_predictions, target_names=mlb.classes_, zero_division=0))


LinearSVC multilabel results:
                                   precision    recall  f1-score   support

                      Exclusivity       0.29      0.40      0.33         5
                        Insurance       0.73      0.67      0.70        24
          Ip Ownership Assignment       0.50      0.33      0.40         6
 Irrevocable Or Perpetual License       0.00      0.00      0.00         1
Unlimited/All-You-Can-Eat-License       1.00      0.33      0.50         6
               Volume Restriction       0.91      0.43      0.59        23
                Warranty Duration       0.57      0.50      0.53        32

                        micro avg       0.62      0.49      0.55        97
                        macro avg       0.57      0.38      0.44        97
                     weighted avg       0.69      0.49      0.56        97
                      samples avg       0.46      0.49      0.47        97

